In [6]:
from collections import Counter
from typing import Dict, Any, Sequence, Tuple, Optional
import sys
import numpy as np

In [7]:
class LruPolicy:
    def __init__(self, max_size: int):
        self.cur_size = 0
        self.max_size = max_size
        self.cache = {}  # (vid, layer, tile) -> (value, size)
        self.access_order = []  # keys in order of access (oldest first)
    
    def get(self, key):
        if key not in self.cache:
            return None

        self.access_order.remove(key)
        self.access_order.append(key)
        
        return self.cache[key][0]
    
    def put(self, key, value, size: int):
        evicted = []
        
        if key in self.cache:
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            self.access_order.remove(key)
            del self.cache[key]
        
        while self.cur_size + size > self.max_size and self.access_order:
            lru_key = self.access_order.pop(0)  # Remove oldest (least recently used)
            lru_size = self.cache[lru_key][1]
            self.cur_size -= lru_size
            del self.cache[lru_key]
            evicted.append(lru_key)
        
        # Add new item if there's space
        if self.cur_size + size <= self.max_size:
            self.cache[key] = (value, size)
            self.access_order.append(key)
            self.cur_size += size
        
        return evicted
    
    def contains(self, key) -> bool:
        return key in self.cache
    
    def remove(self, key):
        if key not in self.cache:
            return False
        
        size = self.cache[key][1]
        self.cur_size -= size
        self.access_order.remove(key)
        del self.cache[key]

        return True
    
    def clear(self):
        """Clear all items from cache."""
        self.cache.clear()
        self.access_order.clear()
        self.cur_size = 0
    
    def get_stats(self) -> Dict[str, Any]:
        """Get cache statistics."""
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

In [ ]:
class CacheEngineEnv:
    def __init__(
        self, 
        n_tiles: int = 16,
        n_layers: int = 2,
        n_videos: int = 100,
        cache_capacity: float = 100e6,  # capacity in bytes
    ):
        self.max_capacity = cache_capacity
        self.tile_size_bytes = {
            0: 2e6 / n_tiles,   # base layer tile size in bytes
            1: 15e6 / n_tiles   # enhancement layer tile size in bytes
        }
        self.n_layers = n_layers
        self.n_videos = n_videos
        self.n_tiles = n_tiles
        
        # Initialize LRU policy
        self.policy = LruPolicy(max_size=int(cache_capacity))

    def _cache_tile(self, vid, layer, tile_idx):
        key = (vid, layer, tile_idx)
        tile_size = self.tile_size_bytes[layer]
        
        # Use LRU policy - automatically handles eviction
        evicted = self.policy.put(key, None, int(tile_size))
        return evicted
    
    def get_tile(self, vid, layer, tile_idx):
        key = (vid, layer, tile_idx)
        if self.policy.contains(key):
            self.policy.get(key)  # Update access order
            return True
        return False
    
    def get_current_capacity(self):
        return self.policy.cur_size
    
    def get_cache_matrix(self):
        cache_matrix = np.zeros((self.n_videos, self.n_layers, self.n_tiles), dtype=np.int8)
        
        # Regenerate matrix from LRU cache
        for (vid, layer, tile_idx) in self.policy.cache.keys():
            cache_matrix[vid, layer, tile_idx] = 1
        
        return cache_matrix
    
    def clear_cache(self):
        self.policy.clear()
    
    def process_cache_prefetching(self, cache_actions):
        self.clear_cache()
        
        # Cache base layer tiles by video popularity
        vid_counts = Counter(act.get('video') for act in cache_actions)
        for vid, _ in vid_counts.most_common():
            for idx in range(self.n_tiles):
                self._cache_tile(vid, 0, idx)
        
        # Cache enhancement layer tiles by request frequency
        enh_counts = Counter()
        for act in cache_actions:
            vid = act['video']
            tiles = act['tiles']
            for idx, tile in enumerate(tiles):
                if tile == 0:
                    continue
                enh_counts[(vid, idx)] += 1
                
        for (vid, tile), _ in enh_counts.most_common():
            self._cache_tile(vid, 1, tile)

        return self.get_cache_matrix()
    
    def reset(self, **kwargs):
        self.clear_cache()
        return None, {"cache": 0}

In [9]:
# Test LRU Policy with tuple keys (video, layer, tile)
if __name__ == "__main__":
    print("=" * 5, "LRU Cache Policy Test", "=" * 5)
    
    # Create LRU cache with 10 MB capacity
    lru = LruPolicy(max_size=10 * 1024 * 1024)  # 10 MB
    
    print(f"Initial state: {lru.get_stats()}\n")
    
    # Add some items using tuple keys (video, layer, tile)
    print("Adding items:")
    evicted = lru.put(
        (0, 0, 0), 
        "data_v0_l0_t0", 
        2 * 1024 * 1024
    )  # 2 MB
    print(f"  Added (0,0,0) (2 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (1, 0, 0), 
        "data_v1_l0_t0", 
        3 * 1024 * 1024
    )  # 3 MB
    print(f"  Added (1,0,0) (3 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (2, 0, 0), 
        "data_v2_l0_t0", 
        4 * 1024 * 1024
    )  # 4 MB
    print(f"  Added (2,0,0) (4 MB), evicted: {evicted}")
    
    print(f"\nCurrent state: {lru.get_stats()}")
    print(f"Access order (oldest→newest): {lru.access_order}\n")
    
    # Access tile (0,0,0) (moves it to most recent)
    print("Accessing (0,0,0)...")
    data = lru.get((0, 0, 0))
    print(f"  Retrieved: {data}")
    print(f"  New access order: {lru.access_order}\n")
    
    # Add item that requires eviction
    print("Adding large item (3 MB) - should evict LRU item...")
    evicted = lru.put((3, 0, 0), "data_v3_l0_t0", 3 * 1024 * 1024)
    print(f"  Evicted: {evicted}")
    print(f"  Current access order: {lru.access_order}")
    print(f"  State: {lru.get_stats()}\n")
    
    # Test contains
    print("Testing contains:")
    for key in [(0, 0, 0), (1, 0, 0), (2, 0, 0), (3, 0, 0)]:
        print(f"  {key}: {lru.contains(key)}")
    
    print("\n" + "=" * 60)

===== LRU Cache Policy Test =====
Initial state: {'size': 0, 'max_size': 10485760, 'num_items': 0, 'utilization': 0.0}

Adding items:
  Added (0,0,0) (2 MB), evicted: []
  Added (1,0,0) (3 MB), evicted: []
  Added (2,0,0) (4 MB), evicted: []

Current state: {'size': 9437184, 'max_size': 10485760, 'num_items': 3, 'utilization': 0.9}
Access order (oldest→newest): [(0, 0, 0), (1, 0, 0), (2, 0, 0)]

Accessing (0,0,0)...
  Retrieved: data_v0_l0_t0
  New access order: [(1, 0, 0), (2, 0, 0), (0, 0, 0)]

Adding large item (3 MB) - should evict LRU item...
  Evicted: [(1, 0, 0)]
  Current access order: [(2, 0, 0), (0, 0, 0), (3, 0, 0)]
  State: {'size': 9437184, 'max_size': 10485760, 'num_items': 3, 'utilization': 0.9}

Testing contains:
  (0, 0, 0): True
  (1, 0, 0): False
  (2, 0, 0): True
  (3, 0, 0): True



In [10]:
if __name__ == "__main__":
    # Test configuration
    n_tiles = 4
    n_layers = 2
    n_videos = 10
    cache_capacity = 100e6  # 30 MB
    
    print("=" * 5, "Cache Storage Test with LRU Policy (Computed Matrix)", "=" * 5)
    
    # Initialize cache environment with LRU
    cache = CacheEngineEnv(
        n_tiles=n_tiles * n_tiles,
        n_layers=n_layers,
        n_videos=n_videos,
        cache_capacity=cache_capacity
    )
    
    print(f"Configuration:")
    print(f"  Grid size: {n_tiles}x{n_tiles} = {n_tiles*n_tiles} tiles")
    print(f"  Layers: {n_layers}")
    print(f"  Videos: {n_videos}")
    print(f"  Capacity: {cache_capacity/1e6:.1f} MB")
    print(f"  Base layer tile size: {cache.tile_size_bytes[0]/1e6:.3f} MB")
    print(f"  Enhancement tile size: {cache.tile_size_bytes[1]/1e6:.3f} MB")
    print(f"  Using LRU: {cache.policy is not None}")
    print()
    
    # Create sample cache actions (prefetch decisions)
    cache_actions = [
        {
            'video': 0,
            'tiles': np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
        {
            'video': 1,
            'tiles': np.array([0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
        {
            'video': 0,
            'tiles': np.array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
    ]
    
    print(f"Processing {len(cache_actions)} cache actions...")
    print(f"Video requests: {[act['video'] for act in cache_actions]}\n")
    
    # Process cache prefetching
    cache_matrix = cache.process_cache_prefetching(cache_actions)
    
    print("Cache Results:")
    current_size = cache.get_current_capacity()
    print(f"  Final capacity used: {current_size/1e6:.2f} MB / {cache.max_capacity/1e6:.1f} MB")
    print(f"  Utilization: {(current_size/cache.max_capacity)*100:.1f}%")
    
    if cache.policy:
        stats = cache.policy.get_stats()
        print(f"  LRU stats: {stats['num_items']} items, {stats['utilization']:.1%} full")
    print()
    
    # Show cache content per video
    print("Cached tiles by video:")
    for vid in range(n_videos):
        base_cached = np.sum(cache_matrix[vid, 0, :])
        enh_cached = np.sum(cache_matrix[vid, 1, :])
        if base_cached > 0 or enh_cached > 0:
            print(f"  Video {vid}: Base={base_cached}/{n_tiles*n_tiles}, Enh={enh_cached}/{n_tiles*n_tiles}")
            if base_cached > 0:
                print(f"    Base layer tiles: {np.where(cache_matrix[vid, 0, :] == 1)[0].tolist()}")
            if enh_cached > 0:
                print(f"    Enh layer tiles:  {np.where(cache_matrix[vid, 1, :] == 1)[0].tolist()}")
    print()
    
    print("Testing tile access (updates LRU):")
    print(f"  Accessing tile (0, 0, 5): {cache.get_tile(0, 0, 5)}")
    print(f"  Accessing tile (1, 0, 3): {cache.get_tile(1, 0, 3)}")
    print(f"  Accessing non-cached (5, 0, 0): {cache.get_tile(5, 0, 0)}")

===== Cache Storage Test with LRU Policy (Computed Matrix) =====
Configuration:
  Grid size: 4x4 = 16 tiles
  Layers: 2
  Videos: 10
  Capacity: 100.0 MB
  Base layer tile size: 0.125 MB
  Enhancement tile size: 0.938 MB
  Using LRU: True

Processing 3 cache actions...
Video requests: [0, 1, 0]

Cache Results:
  Final capacity used: 10.56 MB / 100.0 MB
  Utilization: 10.6%
  LRU stats: 39 items, 10.6% full

Cached tiles by video:
  Video 0: Base=16/16, Enh=4/16
    Base layer tiles: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    Enh layer tiles:  [0, 1, 2, 4]
  Video 1: Base=16/16, Enh=3/16
    Base layer tiles: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    Enh layer tiles:  [1, 2, 5]

Testing tile access (updates LRU):
  Accessing tile (0, 0, 5): True
  Accessing tile (1, 0, 3): True
  Accessing non-cached (5, 0, 0): False
